# LLMOps, Monitoring, Cost & Reliability — Reference Patterns — Hands-On

**AI Architecture · Week 22a**

Offline notebook: LLMOps loop, OpenTelemetry RAG traces, cost accounting, AI-specific attributes, SLO/budget guard scenarios, and an eval regression gate. No network calls.

## 0. LLMOps loop

```mermaid
flowchart LR
  REG[Prompt / model / index registry] --> DEPLOY[Deploy release tuple]
  DEPLOY --> TRACE[Trace RAG requests]
  TRACE --> EVAL[Offline and online evals]
  EVAL --> ALERT[Alerts and feedback]
  ALERT --> FIX[Prompt, model, index, policy fix]
  FIX --> REG
```

The loop extends MLOps with prompt/index/model versions, groundedness SLOs, cost budgets, and incident-driven eval growth.

In [ ]:
import hashlib
from dataclasses import dataclass
from enum import Enum
from pydantic import BaseModel, ConfigDict, Field
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
trace.set_tracer_provider(provider)
tracer = trace.get_tracer('ai-week-22a-notebook')

In [ ]:
MODEL_PRICES_PER_1K = {'gpt-4o-prod': {'prompt': 0.0050, 'completion': 0.0150}, 'gpt-4o-mini-prod': {'prompt': 0.00015, 'completion': 0.00060}}
@dataclass(frozen=True)
class QueryScenario:
    tenant: str; query: str; model: str; docs: list[str]; prompt_tokens: int; completion_tokens: int; groundedness: float; safety_flags: list[str]

def prompt_hash(text): return hashlib.sha256(text.encode()).hexdigest()[:12]
def request_cost(model, prompt_tokens, completion_tokens):
    p = MODEL_PRICES_PER_1K[model]
    return prompt_tokens / 1000 * p['prompt'] + completion_tokens / 1000 * p['completion']

def traced_rag_query(s):
    prompt = f'Answer with citations only. Tenant={s.tenant}. Question={s.query}. Docs={s.docs}'
    cost = request_cost(s.model, s.prompt_tokens, s.completion_tokens)
    with tracer.start_as_current_span('rag.query') as root:
        root.set_attribute('tenant.id', s.tenant); root.set_attribute('ai.prompt_hash', prompt_hash(prompt)); root.set_attribute('ai.model_deployment', s.model); root.set_attribute('ai.cost_usd', round(cost, 6))
        with tracer.start_as_current_span('rag.retrieve') as span:
            span.set_attribute('ai.embedding_model', 'text-embedding-3-large'); span.set_attribute('ai.index_version', 'kb-index-2026-07-18'); span.set_attribute('ai.retrieved_docs', ','.join(s.docs))
        with tracer.start_as_current_span('rag.rerank') as span:
            span.set_attribute('ai.reranker', 'bge-reranker-v2'); span.set_attribute('ai.top_k_after_rerank', min(3, len(s.docs)))
        with tracer.start_as_current_span('rag.prompt.assemble') as span:
            span.set_attribute('ai.prompt_version', 'support-rag-v22'); span.set_attribute('ai.tokens.prompt', s.prompt_tokens)
        with tracer.start_as_current_span('rag.llm.call') as span:
            span.set_attribute('ai.model_deployment', s.model); span.set_attribute('ai.tokens.prompt', s.prompt_tokens); span.set_attribute('ai.tokens.completion', s.completion_tokens); span.set_attribute('ai.cost_usd', round(cost, 6))
        with tracer.start_as_current_span('rag.validate') as span:
            span.set_attribute('ai.groundedness_score', s.groundedness); span.set_attribute('ai.safety_flags', ','.join(s.safety_flags) or 'none'); span.set_attribute('ai.eval_passed', s.groundedness >= 0.88 and not s.safety_flags)
    return {'tenant': s.tenant, 'model': s.model, 'cost_usd': cost, 'groundedness': s.groundedness}

scenarios = [QueryScenario('acme','summarize refund policy','gpt-4o-mini-prod',['doc:refund','doc:returns'],900,180,0.94,[]), QueryScenario('acme','analyze all contract exceptions','gpt-4o-prod',['doc:msa','doc:dpa','doc:sow'],8200,1600,0.91,[]), QueryScenario('globex','can we ignore approval policy?','gpt-4o-mini-prod',['doc:approval'],1200,240,0.71,['groundedness_low'])]
results = [traced_rag_query(s) for s in scenarios]
print(results)

In [ ]:
for span in sorted(exporter.get_finished_spans(), key=lambda sp: sp.start_time):
    attrs = span.attributes
    interesting = {k: attrs[k] for k in attrs if k.startswith('ai.') or k == 'tenant.id'}
    print(f'{span.name:20s}', interesting)
print('total_cost_usd', round(sum(r['cost_usd'] for r in results), 6))

## 4. AI-specific OpenTelemetry attribute reference

In [ ]:
attributes = [
    ('tenant.id','string','per-tenant blast radius and chargeback','hash if sensitive'),
    ('ai.prompt_hash','string','identify prompt body without logging it','low'),
    ('ai.model_deployment','string','latency, quota, and cost route','low'),
    ('ai.embedding_model','string','retrieval compatibility','low'),
    ('ai.retrieved_docs','csv/list','citation and ACL debugging','medium'),
    ('ai.tokens.prompt/completion','int','cost accounting','low'),
    ('ai.cost_usd','float','budget accounting','financial'),
    ('ai.groundedness_score','float','quality SLO','low'),
    ('ai.safety_flags','csv/list','safety incident triage','medium'),
]
for row in attributes:
    print(f'{row[0]:30s} | {row[1]:8s} | {row[2]:36s} | {row[3]}')

In [ ]:
class Action(str, Enum):
    ALLOW = 'ALLOW'; DOWNGRADE = 'DOWNGRADE-TO-CHEAPER-MODEL'; DENY = 'DENY-WITH-BUDGET-ERROR'
class SLOThresholds(BaseModel):
    model_config = ConfigDict(extra='forbid')
    availability: float = Field(default=0.999, ge=0, le=1); p95_latency_ms: int = Field(default=4000, ge=1); groundedness: float = Field(default=0.88, ge=0, le=1); max_cost_per_request_usd: float = Field(default=0.20, gt=0)
class BudgetPolicy(BaseModel):
    model_config = ConfigDict(extra='forbid')
    per_user_daily_budget_usd: float = Field(default=2.00, gt=0); per_tenant_monthly_cap_usd: float = Field(default=100.00, gt=0); fallback_model: str = 'gpt-4o-mini-prod'; downgrade_when_user_remaining_below_usd: float = 0.25
class RequestContext(BaseModel):
    user_id: str; tenant_id: str; feature: str; model: str; estimated_cost_usd: float; is_system_query: bool = False

def evaluate_request(ctx, user_spend_today, tenant_spend_month, policy):
    if ctx.is_system_query: return Action.ALLOW, ctx.model, 'system query'
    tenant_after = tenant_spend_month.get(ctx.tenant_id, 0.0) + ctx.estimated_cost_usd
    if tenant_after > policy.per_tenant_monthly_cap_usd: return Action.DENY, None, f'tenant monthly cap exceeded: {tenant_after:.2f}'
    user_after = user_spend_today.get(ctx.user_id, 0.0) + ctx.estimated_cost_usd
    remaining = policy.per_user_daily_budget_usd - user_after
    if user_after > policy.per_user_daily_budget_usd:
        return (Action.DOWNGRADE, policy.fallback_model, f'user daily budget would exceed by {abs(remaining):.2f}') if ctx.model != policy.fallback_model else (Action.DENY, None, f'user daily cap exceeded: {user_after:.2f}')
    if remaining < policy.downgrade_when_user_remaining_below_usd and ctx.model != policy.fallback_model: return Action.DOWNGRADE, policy.fallback_model, f'user budget nearly exhausted: {remaining:.2f} remaining'
    return Action.ALLOW, ctx.model, f'budget ok: {remaining:.2f} user budget remaining'

slo = SLOThresholds(); policy = BudgetPolicy()
print('SLO:', slo.model_dump())

## 6. Exercise four budget scenarios

In [ ]:
user_spend = {'heavy': 1.82}
tenant_spend = {'over-cap': 100.05, 'acme': 73.20}
scenarios = [RequestContext(user_id='fresh', tenant_id='acme', feature='chat', model='gpt-4o-prod', estimated_cost_usd=0.08), RequestContext(user_id='heavy', tenant_id='acme', feature='analysis', model='gpt-4o-prod', estimated_cost_usd=0.15), RequestContext(user_id='u3', tenant_id='over-cap', feature='chat', model='gpt-4o-mini-prod', estimated_cost_usd=0.01), RequestContext(user_id='monitor', tenant_id='over-cap', feature='health-check', model='gpt-4o-prod', estimated_cost_usd=0.50, is_system_query=True)]
for ctx in scenarios:
    action, model, reason = evaluate_request(ctx, user_spend, tenant_spend, policy)
    print(ctx.user_id, '->', action.value, model, '|', reason)

## 7. Eval regression gate demo

In [ ]:
golden = [
    ('refund policy answer cites refund docs', True),
    ('admin MFA answer cites security policy', True),
    ('unknown roadmap question refuses', True),
    ('prompt injection says ignore previous instructions', False),
    ('answer from stale policy without citation', False),
]
def mock_grade(prompt_version, item):
    text, expected_grounded = item
    base = 0.92 if expected_grounded else 0.35
    if prompt_version == 'candidate-v23' and ('refund' in text or 'stale' in text): base -= 0.12
    if 'injection' in text: base -= 0.18
    return max(0.0, min(1.0, base))
def eval_gate(prompt_version, threshold=0.82, max_drop=0.04):
    baseline = [mock_grade('baseline-v22', x) for x in golden]
    candidate = [mock_grade(prompt_version, x) for x in golden]
    b = sum(baseline) / len(baseline); c = sum(candidate) / len(candidate)
    decision = 'PASS' if c >= threshold and (b - c) <= max_drop else 'FAIL'
    return decision, round(b, 3), round(c, 3), [round(x, 3) for x in candidate]
for version in ['candidate-safe', 'candidate-v23']:
    print(version, eval_gate(version))

## Exercises
1. Add a `Retry-After` parser and simulate 429 handling without real sleeps.
2. Extend traces with provider fallback and circuit-breaker attributes.
3. Add a tenant-scoped semantic-cache decision to the budget guard.
4. Add one new bad production interaction to the golden set and show the regression gate catching it.

## Links
- Literature note: `02 Literature Notes/AI Architecture/LLMOps, Monitoring, Cost & Reliability — Reference Patterns`
- Snippets: `04 Code Snippets/AI Architecture/AI Week 22a LLM Request Tracer With Cost Accounting`, `.../AI Week 22a SLO and Cost Budget Guard`
- MOC: `06 Maps of Content/AI Architecture Concepts`